In [ ]:
import os
import platform
import subprocess

print("Operating system:", platform.system())
print("Python version:", platform.python_version())

try:
    result = subprocess.run(
        ["nvidia-smi"],  #reports gpu information
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("\nNVIDIA GPU detected:")
        print(result.stdout)
    else:
        print("\nNo accessible NVIDIA GPU detected.")
except FileNotFoundError:
    print("\nNVIDIA tools are not available in this runtime.")

Operating system: Linux
Python version: 3.13.15

NVIDIA GPU detected:
Tue Oct 20 08:34:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |  

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
%%writefile first_cuda.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// This function runs on the GPU
__global__ void vectorAdd(int *A, int *B, int *C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    const int N = 10;
    int A[N], B[N], C[N];

    for (int i = 0; i < N; i++) {
        A[i] = i;
        B[i] = i * 2;
    }

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, N * sizeof(int));
    cudaMalloc(&d_B, N * sizeof(int));
    cudaMalloc(&d_C, N * sizeof(int));

    cudaMemcpy(d_A, A, N * sizeof(int),
               cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, N * sizeof(int),
               cudaMemcpyHostToDevice);

    vectorAdd<<<1, 256>>>(d_A, d_B, d_C, N);

    cudaMemcpy(C, d_C, N * sizeof(int),
               cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++) {
        cout << A[i] << " + " << B[i]
             << " = " << C[i] << endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing first_cuda.cu


In [ ]:
!nvcc first_cuda.cu -o first_cuda
!./first_cuda

0 + 0 = 0
1 + 2 = 3
2 + 4 = 6
3 + 6 = 9
4 + 8 = 12
5 + 10 = 15
6 + 12 = 18
7 + 14 = 21
8 + 16 = 24
9 + 18 = 27


In [ ]:
%%writefile cpu_benchmark.cpp
#include <iostream>
#include <vector>
#include <chrono>

using namespace std;
using namespace chrono;

int main() {
    const int N = 1000000;

    vector<int> A(N, 10);
    vector<int> B(N, 20);
    vector<int> C(N);

    auto start = high_resolution_clock::now();

    for (int i = 0; i < N; i++) {
        C[i] = A[i] + B[i];
    }

    auto stop = high_resolution_clock::now();

    auto time_taken =
        duration<double, milli>(stop - start).count();

    cout << "First result: " << C[0] << endl;
    cout << "Last result: " << C[N - 1] << endl;
    cout << "Execution time: " << time_taken << " ms" << endl;

    return 0;
}

Writing cpu_benchmark.cpp


In [ ]:
!g++ -O2 cpu_benchmark.cpp -o cpu_benchmark
!./cpu_benchmark

First result: 30
Last result: 30
Execution time: 0.537565 ms


In [ ]:
%%writefile vector_benchmark.cu
#include <iostream>
#include <vector>
#include <chrono>
#include <cuda_runtime.h>

using namespace std;
using namespace chrono;

__global__ void vectorAdd(
    const float* A, const float* B, float* C, int N
) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    const int N = 1000000;
    size_t bytes = N * sizeof(float);

    vector<float> A(N, 10.0f);
    vector<float> B(N, 20.0f);
    vector<float> C_cpu(N);
    vector<float> C_gpu(N);

    // CPU benchmark
    auto cpu_start = high_resolution_clock::now();

    for (int i = 0; i < N; i++) {
        C_cpu[i] = A[i] + B[i];
    }

    auto cpu_stop = high_resolution_clock::now();

    double cpu_ms =
        duration<double, milli>(cpu_stop - cpu_start).count();

    float *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    // GPU benchmark, including memory transfers
    cudaDeviceSynchronize();
    auto gpu_start = high_resolution_clock::now();

    cudaMemcpy(d_A, A.data(), bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B.data(), bytes, cudaMemcpyHostToDevice);

    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    vectorAdd<<<blocks, threads>>>(d_A, d_B, d_C, N);

    cudaError_t error = cudaGetLastError();
    if (error != cudaSuccess) {
        cerr << "Kernel launch error: "
             << cudaGetErrorString(error) << endl;
        return 1;
    }

    cudaDeviceSynchronize();

    cudaMemcpy(C_gpu.data(), d_C, bytes, cudaMemcpyDeviceToHost);

    auto gpu_stop = high_resolution_clock::now();

    double gpu_ms =
        duration<double, milli>(gpu_stop - gpu_start).count();

    bool correct = true;

    for (int i = 0; i < N; i++) {
        if (C_cpu[i] != C_gpu[i]) {
            correct = false;
            break;
        }
    }

    cout << "\n--- Vector Addition Benchmark ---\n";
    cout << "Elements: " << N << '\n';
    cout << "CPU time: " << cpu_ms << " ms\n";
    cout << "GPU total time: " << gpu_ms << " ms\n";
    cout << "Results correct: " << (correct ? "Yes" : "No") << '\n';

    if (gpu_ms > 0) {
        cout << "CPU/GPU total-time ratio: "
             << cpu_ms / gpu_ms << "x\n";
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return correct ? 0 : 1;
}

Overwriting vector_benchmark.cu


In [ ]:
!nvcc -O2 vector_benchmark.cu -o vector_benchmark
!./vector_benchmark


--- Vector Addition Benchmark ---
Elements: 1000000
CPU time: 0.770237 ms
GPU total time: 3.0974 ms
Results correct: Yes
CPU/GPU total-time ratio: 0.248672x
